In [ ]:
from chembl_webresource_client.new_client import new_client
from chembl_webresource_client.settings import Settings
import pandas as pd

Settings.Instance().CACHING = False

activity = new_client.activity
molecule = new_client.molecule
mechanism = new_client.mechanism

target_real = 'CHEMBL287'

# ── STEP 1: Get all quantitative binding records ──────────────────────────
activities = activity.filter(
    target_chembl_id      = target_real,
    standard_type__in     = ['Ki', 'IC50'],
    pchembl_value__isnull = False
).only(
    'molecule_chembl_id',
    'molecule_pref_name',
    'parent_molecule_chembl_id',
    'pchembl_value',
    'standard_type',
    'standard_value',
    'standard_units'
)

activities_df = pd.DataFrame(activities)
activities_df['pchembl_value'] = pd.to_numeric(
    activities_df['pchembl_value'], errors='coerce'
)
activities_df.to_csv('activities3.csv', index=False)
print(f"Total activity records: {len(activities_df)}")

# ── STEP 2: Get unique parent IDs and their mean pChEMBL ─────────────────
parent_ids = activities_df['parent_molecule_chembl_id'].dropna().unique().tolist()

agg_df = (
    activities_df
    .groupby('parent_molecule_chembl_id')
    .agg(
        mean_pchembl = ('pchembl_value', 'mean'),
        best_pchembl = ('pchembl_value', 'max'),
        assay_count  = ('pchembl_value', 'count')
    )
    .reset_index()
    .round({'mean_pchembl': 2, 'best_pchembl': 2})
)

# ── STEP 3: Fetch molecule metadata (Phase, name) ─────────────────────────
all_mols   = []
batch_size = 100

for i in range(0, len(parent_ids), batch_size):
    batch  = parent_ids[i : i + batch_size]
    result = molecule.filter(
        molecule_chembl_id__in = batch
    ).only('molecule_chembl_id', 'max_phase', 'pref_name')
    all_mols.extend(list(result))

mol_df = pd.DataFrame(all_mols)
mol_df.to_csv('molecule3.csv', index=False)

# ── STEP 4: Fetch mechanism labels ───────────────────────────────────────
# We fetch mechanism for ALL parent IDs, not just those already in the
# mechanism table. Drugs missing from mechanism get NaN — that is fine.
# We are using mechanism only to LABEL action_type, not to gate the list.

print("\nFetching mechanism labels...")
all_mec    = []

for i in range(0, len(parent_ids), batch_size):
    batch  = parent_ids[i : i + batch_size]
    result = mechanism.filter(
        molecule_chembl_id__in = batch
    ).only('molecule_chembl_id', 'action_type', 'parent_molecule_chembl_id')
    all_mec.extend(list(result))

mec_df = pd.DataFrame(all_mec)

# If mechanism table is empty for some molecules, handle gracefully
if mec_df.empty:
    mec_df = pd.DataFrame(columns=[
        'molecule_chembl_id', 'action_type', 'parent_molecule_chembl_id'
    ])

# Keep one action_type per parent (deduplicate)
mec_clean = (
    mec_df[['parent_molecule_chembl_id', 'action_type']]
    .drop_duplicates(subset='parent_molecule_chembl_id')
)
mec_df.to_csv('mechanism3.csv', index=False)
print(f"Mechanism records fetched: {len(mec_df)}")

# ── STEP 5: Merge everything together ─────────────────────────────────────
merged_df = (
    agg_df
    .merge(
        mol_df.rename(columns={'molecule_chembl_id': 'parent_molecule_chembl_id'}),
        on='parent_molecule_chembl_id', how='left'
    )
    .merge(
        mec_clean,
        on='parent_molecule_chembl_id', how='left'   # LEFT JOIN — keeps all binders
    )
)

merged_df['max_phase']   = pd.to_numeric(merged_df['max_phase'], errors='coerce')
merged_df['action_type'] = merged_df['action_type'].str.upper().str.strip()

print(f"\nAction types present in dataset:")
print(merged_df['action_type'].value_counts(dropna=False))

# ── STEP 6: Filter to AGONISTS and MODULATORS only ───────────────────────
# These action types activate or positively modulate sigma-1R,
# which is the desired mechanism for neuroprotection and repurposing.
# ANTAGONISTS and INHIBITORS are excluded as they block receptor function.

agonist_types = ['AGONIST', 'MODULATOR', 'POSITIVE ALLOSTERIC MODULATOR', 
                 'PARTIAL AGONIST']

agonists_df = merged_df[
    merged_df['action_type'].isin(agonist_types)
].copy()

print(f"\nMolecules with agonist/modulator action type: {len(agonists_df)}")

# ── STEP 7: Filter to Phase 4 only ───────────────────────────────────────
phase4_agonists = (
    agonists_df[agonists_df['max_phase'] == 4]
    .sort_values('mean_pchembl', ascending=False)
    .reset_index(drop=True)
)

print(f"Phase 4 agonists/modulators: {len(phase4_agonists)}")
print("\nFinal ranked list:")
print(phase4_agonists[[
    'pref_name', 'action_type', 'mean_pchembl', 
    'best_pchembl', 'assay_count'
]].to_string(index=False))

phase4_agonists.to_csv('final_candidates.csv', index=False)
print("\nfinal_candidates.csv saved.")


#the 4 manually added  molecules 

# ── KNOWLEDGE-GUIDED CANDIDATES ───────────────────────────────────────────
# The following Phase 4 drugs are manually added based on published
# sigma-1R mechanistic evidence. They are confirmed binders in activities3.csv
# but do not appear in the ChEMBL mechanism table under sigma-1R because
# their PRIMARY approved mechanism is a different target. Their sigma-1R
# engagement has been independently validated in peer-reviewed literature
# and clinical studies, making them the strongest repurposing candidates.

knowledge_guided = pd.DataFrame([
    {
        'pref_name'               : 'PENTAZOCINE',
        'parent_molecule_chembl_id': 'CHEMBL60542',
        'max_phase'               : 4,
        'action_type'             : 'AGONIST',
        'selection_basis'         : 'Prototypical sigma-1R reference agonist; '
                                    'human crystal structure solved (PDB: 6DK1)',
        'key_reference'           : 'Huang et al., 2017, Cell'
    },
    {
        'pref_name'               : 'DEXTROMETHORPHAN',
        'parent_molecule_chembl_id': 'CHEMBL52440',
        'max_phase'               : 4,
        'action_type'             : 'AGONIST',
        'selection_basis'         : 'Sigma-1R agonism (Ki=142-652nM) mediates '
                                    'neuroprotection independently of NMDA antagonism',
        'key_reference'           : 'Nguyen et al., 2014, Trends Pharmacol Sci'
    },
    {
        'pref_name'               : 'DONEPEZIL',
        'parent_molecule_chembl_id': 'CHEMBL502',
        'max_phase'               : 4,
        'action_type'             : 'AGONIST',
        'selection_basis'         : 'IC50=14.6nM at sigma-1R; 93% receptor occupancy '
                                    'confirmed by PET imaging at therapeutic doses',
        'key_reference'           : 'Meunier et al., 2006, Br J Pharmacol'
    },
    {
        'pref_name'               : 'FLUVOXAMINE',
        'parent_molecule_chembl_id': 'CHEMBL814',
        'max_phase'               : 4,
        'action_type'             : 'AGONIST',
        'selection_basis'         : 'Highest sigma-1R affinity among all SSRIs '
                                    '(Ki=36nM); confirmed by human PET study; '
                                    'clinical trial conducted specifically for sigma-1R',
        'key_reference'           : 'Hashimoto et al., 2009, Biol Psychiatry; '
                                    'TOGETHER trial 2021'
    },
])

# Merge pChEMBL values from your activities data onto these candidates
# so they carry the same quantitative data as the pipeline-derived results
pchembl_lookup = agg_df.set_index('parent_molecule_chembl_id')[
    ['mean_pchembl', 'best_pchembl', 'assay_count']
]

knowledge_guided = knowledge_guided.merge(
    pchembl_lookup,
    on  = 'parent_molecule_chembl_id',
    how = 'left'
)

print("Knowledge-guided candidates with ChEMBL binding confirmation:")
print(knowledge_guided[[
    'pref_name', 'mean_pchembl', 'best_pchembl', 
    'assay_count', 'selection_basis'
]].to_string(index=False))

knowledge_guided.to_csv('knowledge_guided_candidates.csv', index=False)




# merging both files together 


# ── At this point you have two separate dataframes ────────────────────────
# phase4_agonists     → what the pipeline returned (fentanyl, raloxifene etc.)
# knowledge_guided    → your manually added four drugs

# ── STEP 1: Make sure both have the same columns before merging ───────────
# Add the selection_basis and key_reference columns to pipeline results
# Pipeline drugs don't have a literature reference so we label them clearly

phase4_agonists['selection_basis'] = 'Identified via ChEMBL quantitative binding pipeline (Ki/IC50, Phase 4, agonist/modulator filter)'
phase4_agonists['key_reference']   = 'ChEMBL Database (CHEMBL287 activity records)'

# ── STEP 2: Confirm both dataframes have matching column names ─────────────
print("Pipeline columns      :", phase4_agonists.columns.tolist())
print("Knowledge guided cols :", knowledge_guided.columns.tolist())

# ── STEP 3: Concatenate both into one master dataframe ────────────────────
# pd.concat stacks rows on top of each other — this is correct here because
# both dataframes have the same columns and you are just combining the rows

master_df = pd.concat(
    [knowledge_guided, phase4_agonists],
    ignore_index = True    # resets index from 0 so there are no duplicate indices
)

# ── STEP 4: Remove duplicates in case any drug appears in both groups ──────
# A drug like pentazocine might appear in both if the pipeline also caught it.
# Keep the knowledge_guided version (which appears first) since it has the 
# richer selection_basis and key_reference information.

master_df = master_df.drop_duplicates(
    subset = 'parent_molecule_chembl_id',
    keep   = 'first'    # keeps knowledge_guided version over pipeline version
)

# ── STEP 5: Sort by mean pChEMBL so the best binders appear at the top ────
master_df = master_df.sort_values(
    'mean_pchembl', ascending=False
).reset_index(drop=True)

# ── STEP 6: Add a source column so you can always tell where each came from
master_df['source'] = master_df['key_reference'].apply(
    lambda x: 'Knowledge Guided' if 'Hashimoto' in str(x) 
              or 'Meunier' in str(x) 
              or 'Nguyen' in str(x) 
              or 'Huang' in str(x)
              else 'Pipeline Derived'
)

print("\nMaster candidate dataframe:")
print(master_df[[
    'pref_name', 'mean_pchembl', 'best_pchembl', 
    'assay_count', 'action_type', 'source'
]].to_string(index=False))

master_df.to_csv('master_candidates.csv', index=False)
print(f"\nmaster_candidates.csv saved — {len(master_df)} total candidates")

# ── STEP 7: Apply your 3 elimination rules to get the final 5 ─────────────
final_5 = master_df[
    # Rule 1 — pChEMBL must be >= 6
    (master_df['mean_pchembl'] >= 6) &
    
    # Rule 2 and 3 — exclude the contradicted and weak ones
    (~master_df['pref_name'].isin([
        'FENTANYL',       # literature directly contradicts ChEMBL data
        'BREXPIPRAZOLE',  # single assay + contradicted by published review
        'TESTOSTERONE',   # single assay, no mechanistic evidence
        'LASMIDITAN',     # below threshold, no sigma-1R mechanistic basis
        'RAMELTEON'       # far below threshold, nonspecific hit
    ])) &
    
    # Pentazocine is the standard, not a candidate
    (~master_df['pref_name'].isin(['PENTAZOCINE', '(+)-PENTAZOCINE']))
    
].sort_values('mean_pchembl', ascending=False).head(6).reset_index(drop=True)

print("\n" + "="*60)
print("FINAL 5 DOCKING CANDIDATES")
print("="*60)
print(final_5[[
    'pref_name', 'mean_pchembl', 'best_pchembl', 
    'assay_count', 'action_type', 'source'
]].to_string(index=False))

final_5.to_csv('final_5_candidates.csv', index=False)
print("\nfinal_5_candidates.csv saved.")






In [7]:
import pandas as pd 
from chembl_webresource_client.new_client import new_client
from chembl_webresource_client.settings import Settings


Settings.Instance().CACHING = False
Settings.Instance().TIMEOUT= 1000

activity = new_client.activity
molecule = new_client.molecule
mechanism = new_client.mechanism


# Read the final 5 candidates
final_candidate = pd.read_csv('final_5_candidates.csv')

# Check what is already in the file
candidate_ids = final_candidate['parent_molecule_chembl_id'].tolist()
print(f'Candidates before adding standard: {len(candidate_ids)}')
print(candidate_ids)

# Build pentazocine row to add — must match all columns in final_candidate
pentazocine_row = pd.DataFrame([{
    'parent_molecule_chembl_id': 'CHEMBL60542',
    'pref_name'                : 'PENTAZOCINE',
    'max_phase'                : 4,
    'action_type'              : 'AGONIST',
    'mean_pchembl'             : None,   # will be filled from agg_df if available
    'best_pchembl'             : None,
    'assay_count'              : None,
    'selection_basis'          : 'Docking reference standard — prototypical SIGMAR1 agonist. Human crystal structure PDB: 6DK1.',
    'key_reference'            : 'Huang et al., 2017, Cell',
    'source'                   : 'Docking Standard'
}])

# Only add pentazocine if it is not already in the file
if 'CHEMBL60542' not in candidate_ids:
    final_candidate = pd.concat(
        [final_candidate, pentazocine_row],
        ignore_index=True
    )
    print('Pentazocine added as docking standard.')
else:
    print('Pentazocine already present.')

# molecule_chembl_id = parent_molecule_chembl_id for all these drugs
# no need to hardcode — just copy the column
final_candidate['molecule_chembl_id'] = final_candidate['parent_molecule_chembl_id']

# Verify
print(f'\nTotal rows including standard: {len(final_candidate)}')
print(final_candidate[['pref_name', 'molecule_chembl_id',
                        'parent_molecule_chembl_id', 'source']].to_string(index=False))

final_candidate.to_csv('final_5_candidates.csv', index=False)
print('\nfinal_5_candidates.csv updated.')


Candidates before adding standard: 4
['CHEMBL502', 'CHEMBL814', 'CHEMBL81', 'CHEMBL52440']


/tmp/ipykernel_1513/3291464915.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_candidate = pd.concat(


Pentazocine added as docking standard.

Total rows including standard: 5
       pref_name molecule_chembl_id parent_molecule_chembl_id           source
       DONEPEZIL          CHEMBL502                 CHEMBL502 Knowledge Guided
     FLUVOXAMINE          CHEMBL814                 CHEMBL814 Knowledge Guided
      RALOXIFENE           CHEMBL81                  CHEMBL81 Pipeline Derived
DEXTROMETHORPHAN        CHEMBL52440               CHEMBL52440 Knowledge Guided
     PENTAZOCINE        CHEMBL60542               CHEMBL60542 Docking Standard

final_5_candidates.csv updated.


In [22]:
#getting the canonical smiles now here 

molecule_read = pd.read_csv('final_5_candidates.csv')
#print(molecule_read)

molecule_list = molecule_read['molecule_chembl_id'].tolist()
#print(molecule_list)

#starting the filter now to get the canonical smiles using the molecule filter method 

mole_res = molecule.filter(molecule_chembl_id__in = molecule_list).only('molecule_chembl_id','pref_name', 'molecule_structures')
mole_df = pd.DataFrame(mole_res)
#print(mole_df)

mole_df['canonical_smiles']= mole_df['molecule_structures'].apply(lambda x : x['canonical_smiles'])
#print(mole_df)

#merging together the two files to make a final one 

molecule_read =  molecule_read.merge(
    mole_df[['molecule_chembl_id', 'canonical_smiles']], 
    on = 'molecule_chembl_id',
    how = 'left'
)

molecule_read.to_csv('molecule_canonical_smiles.csv', index = False )

print(molecule_read[['pref_name', 'canonical_smiles', 'molecule_chembl_id']])

          pref_name                                   canonical_smiles  \
0         DONEPEZIL      COc1cc2c(cc1OC)C(=O)C(CC1CCN(Cc3ccccc3)CC1)C2   
1       FLUVOXAMINE                COCCCC/C(=N\OCCN)c1ccc(C(F)(F)F)cc1   
2        RALOXIFENE  O=C(c1ccc(OCCN2CCCCC2)cc1)c1c(-c2ccc(O)cc2)sc2...   
3  DEXTROMETHORPHAN     COc1ccc2c(c1)[C@]13CCCC[C@@H]1[C@H](C2)N(C)CC3   
4       PENTAZOCINE      CC(C)=CCN1CC[C@]2(C)c3cc(O)ccc3C[C@H]1[C@H]2C   

  molecule_chembl_id  
0          CHEMBL502  
1          CHEMBL814  
2           CHEMBL81  
3        CHEMBL52440  
4        CHEMBL60542  
